 # Table of Contents
+ [Import](#Import_0)
+ [Files](#Files_1)
+ [Input](#Input_2)
+ [Output](#Output_3)
+ [Download ChEBI files](#Download_ChEBI_files_4)
+ [Gather and clean data](#Gather_and_clean_data_5)
+ [Save DB](#Save_DB_6)


<a class="anchor" id="Import_0"></a>
# <span class=title_0 style="color: #4E2C73">Import</span>

In [1]:
import sys
import re 

from pathlib import Path
import urllib.request

sys.path.append('../../')

# Resolve project directories
from file_management import get_files_dir,check_save_file
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

import gzip
import shutil

import pandas as pd

<a class="anchor" id="Files_1"></a>
# <span class=title_0 style="color: #4E2C73">Files</span>

This notebook builds a local ChEBI database by downloading and

processing official flat files from the EBI FTP server.

<a class="anchor" id="Input_2"></a>
# <span class=title_0 style="color: #4E2C73">Input</span>

In [2]:
chebi_names_url = ('https://ftp.ebi.ac.uk/pub/databases/chebi/'
                   'flat_files/names.tsv.gz')

chebi_compounds_url = ('https://ftp.ebi.ac.uk/pub/databases/chebi/'
                       'flat_files/compounds.tsv.gz')


ChEBI_files = INPUT_DIR / 'ChEBI'
ChEBI_files.mkdir(parents=True, exist_ok=True)


<a class="anchor" id="Output_3"></a>
# <span class=title_0 style="color: #4E2C73">Output</span>

In [3]:
# Local directory for ChEBI files
ChEBI_files = INPUT_DIR / 'ChEBI'
ChEBI_files.mkdir(parents=True, exist_ok=True)

# Output database file
OUTPUT_FILE_ChEBI_DB = 'chebi.json'

<a class="anchor" id="Download_ChEBI_files_4"></a>
# <span class=title_0 style="color: #4E2C73">Download ChEBI files</span>

In [4]:
def download_and_gunzip(url: str, out_dir: Path, overwrite: bool = False):
    gz_path = out_dir / Path(url).name
    out_path = gz_path.with_suffix("")  # removes .gz

    # Skip if already exists
    if out_path.exists() and not overwrite:
        print(f"✔ {out_path.name} already exists, skipping.")
        return out_path
    print(url)
    print(f"⬇ Downloading {gz_path.name}...")
    urllib.request.urlretrieve(url, gz_path)

    print(f"📦 Extracting {out_path.name}...")
    with gzip.open(gz_path, "rb") as f_in:
        with open(out_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

    gz_path.unlink()  # remove .gz
    return out_path

In [5]:
# Download required ChEBI flat files

download_and_gunzip(chebi_names_url, ChEBI_files)
download_and_gunzip(chebi_compounds_url, ChEBI_files)

✔ names.tsv already exists, skipping.
✔ compounds.tsv already exists, skipping.


PosixPath('/Users/elisamarquez/Documents/PhD/Utimo/ELISER-StrainDesignDB/files/Input/ChEBI/compounds.tsv')

<a class="anchor" id="Gather_and_clean_data_5"></a>
# <span class=title_0 style="color: #4E2C73">Gather and clean data</span>

In [6]:
# Load the ChEBI database into a Pandas DataFrame.
full_compounds_file = ChEBI_files / 'compounds.tsv'
full_compounds = pd.read_table(full_compounds_file, dtype=str, index_col='id')
full_compounds = full_compounds.loc[:,'name']
full_compounds.head()

id
3     ((R)-3-Hydroxybutanoyl)(n-2)
7                    (+)-car-3-ene
8          (+)-8-hydroxycalamenene
9                     (+)-Adlumine
10            (+)-Atherospermoline
Name: name, dtype: object

In [7]:
full_compounds.shape

(204718,)

In [8]:
# Load the ChEBI database into a Pandas DataFrame.

full_syn_file = ChEBI_files / 'names.tsv'
full_syn = pd.read_table(full_syn_file, dtype=str, index_col='compound_id')
full_syn = full_syn.loc[full_syn.language_code=='en',['type','name']]
full_syn.head()

,type,name
compound_id,,
3,SYNONYM,((R)-3-Hydroxybutanoyl)(n-2)
7,SYNONYM,(<i>S</i>)-(+)-3-carene
7,SYNONYM,"(1<i>S</i>)-3,7,7-trimethylbicyclo[4.1.0]hept-..."
7,SYNONYM,(+)-3-Carene
7,IUPAC NAME,"(1<i>S</i>,6<i>R</i>)-3,7,7-trimethylbicyclo[4..."


In [9]:
full_syn = pd.read_table(full_syn_file, dtype=str, index_col='compound_id')


In [10]:
full_syn.shape

(387829, 7)

In [11]:
duplicates = full_syn.index[full_syn.index.duplicated()]
full_syn.loc[duplicates].head()


,id,name,type,status_id,adapted,language_code,ascii_name
compound_id,,,,,,,
7,61702,(<i>S</i>)-(+)-3-carene,SYNONYM,1,false,en,(S)-(+)-3-carene
7,61700,"(1<i>S</i>)-3,7,7-trimethylbicyclo[4.1.0]hept-...",SYNONYM,1,false,en,"(1S)-3,7,7-trimethylbicyclo[4.1.0]hept-3-ene"
7,16,(+)-3-Carene,SYNONYM,1,false,en,(+)-3-Carene
7,61699,"(1<i>S</i>,6<i>R</i>)-3,7,7-trimethylbicyclo[4...",IUPAC NAME,1,false,en,"(1S,6R)-3,7,7-trimethylbicyclo[4.1.0]hept-3-ene"
7,61701,(1<i>S</i>)-(+)-3-carene,SYNONYM,1,false,en,(1S)-(+)-3-carene


In [12]:
full_syn = full_syn.drop_duplicates()
full_syn = full_syn.groupby('compound_id').apply(lambda x: x['name'].tolist())
full_syn.name = 'Synonym'

In [13]:
ChEBI = pd.concat([full_compounds,full_syn], axis=1)

In [14]:
ChEBI = ChEBI.dropna(how='all')

In [15]:
ChEBI.loc[:,'Full_synonym'] = ChEBI.loc[:,'Synonym']

In [16]:
def clean_chebi_name(synonym_list):
    cleaned_synonyms = []
    for text in synonym_list:
        if isinstance(text, str):
            text = re.sub(r"<[^>]+>", "", text)
            text = re.sub(r"\s+", " ", text).strip()
            cleaned_synonyms.append(text)

    return(cleaned_synonyms)

In [17]:
ChEBI["Synonym"] = ChEBI["Synonym"].apply(
    lambda x: clean_chebi_name(x) if isinstance(x, list) else x
)

In [18]:
ChEBI.shape

(204718, 3)

In [19]:
ChEBI.loc[ChEBI.Synonym.isna(),'Synonym'] = ChEBI.loc[ChEBI.Synonym.isna(),'name']
ChEBI.loc[ChEBI.Full_synonym.isna(),'Full_synonym'] = ChEBI.loc[ChEBI.Full_synonym.isna(),'name']

In [20]:
ChEBI.loc[1447]

name            3-acylpyruvic acid
Synonym         3-acylpyruvic acid
Full_synonym    3-acylpyruvic acid
Name: 1447, dtype: object

<a class="anchor" id="Save_DB_6"></a>
# <span class=title_0 style="color: #4E2C73">Save DB</span>

In [22]:
_ = check_save_file(
    ChEBI,
    OUTPUT_FILE_ChEBI_DB,
    "ChEBI",
    input_dir=True
)

Saved file in: /Users/elisamarquez/Documents/PhD/Utimo/ELISER-StrainDesignDB/files/Input/ChEBI/chebi.json
